# Frameworks for fine-tuning

Now that we know more about what it means to fine-tune, we can see how to do it practically. Most modern deep learning developments are, one way or another, based on [Pytorch](https://pytorch.org). In principle, nothing would stop us from implementing a fine-tuning pipeline in pure Pytorch. However, there would be a lot of boilerplate code to be written; moreover, if we decided that our workload needs multiple GPUs, vanilla Pytorch would require some level of code rewriting. To avoid this, a number of frameworks that abstract over Pytorch have risen, such as [Pytorch lightning](https://lightning.ai/docs/pytorch/stable) and Hugging Face (HF) [accelerate](https://github.com/huggingface/accelerate) with the rest of the HF [stack](https://github.com/orgs/huggingface/repositories). 

Lightning is very generic and can be used to wrap any neural network architecture. The HF libraries, instead, are mainly meant to be used with transformers (such as LLMs) and provide a tight integration with the [hub](https://github.com/huggingface/huggingface_hub) for model downloading and sharing, [trl](https://github.com/huggingface/trl) for reinforcement learning, [tokenizers](https://github.com/huggingface/tokenizers) and [peft](https://github.com/huggingface/peft) for parameter-efficient finetuning (such as LoRA), among many others. Together, they constitute a tightly integrated ecosystem which simplifies life for both research and production for people working with LLMs. Thus, we will focus on its usage!

## The Hugging Face fine-tuning stack

The HF ecosystem is deliberately modular, and different libraries deal with different parts of the training workflow. For our fine-tuning example, we will deal with the following:

- Transformers: used to load the models, their tokenizers and relative configuration
- PEFT: used to implement parameter-efficient strategies, such as LoRA, QLoRA, adapters, etc.
- TRL: used to implement the actual training loop, implementing algorithms such as supervised fine-tuning (SFT) and some RL algorithms (DPO, PPO, GRPO, etc.)
- Accelerate: used to abstract the underlying hardware and parallelisation strategies (more on that later)

Our fine-tuning script will simply combine these building blocks :)

### Transformers: models and tokenizers
The [transformers](https://github.com/huggingface/transformers) library provides means to load pretrained models and tokenizers, as well as potentially managing their distribution to different hardware, the precision at which to load them, and lots more. In our case, let us use a small Qwen 1.5B model as base:
```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
```
At this stage, we have just loaded the model, but we have not said anything about the training method nor distribution strategy. A small note on the `use_fast` parameter: some tokenizers have been implemented by the HF developers in Rust and reach very good performance. If such a tokenizer is available for the chosen model, `use_fast=True` will load it!

### PEFT: Parameter-Efficient Fine-Tuning

As discussed earlier, training all parameters of an LLM can be expensive, thus it is often advised to resort to techniques like (Q)LoRA. The `peft` library allows us to do it in the following way:
```python
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj','k_proj','v_proj','o_proj','fc_in','fc_out'])